v0.1  
**Videoglancer.top - Convert YouTube video to PDF/PPTX**

**Instructions (Very Simple!)**

1. Run the 3 blocks below in order — just click ▶️ on each
2. Wait for the temporary Cloudflare Tunnel URL to appear in Block 3
3. Click that tunnel link to open the website

---

**⚠️ Important Notes:**
- Keep this notebook running (don't close the tab)
- The session will timeout after 90 minutes of inactivity
- Files are lost when Colab runtime resets. Only Google Drive saves persist.
- Anyone with the tunnel URL can access the app. Don't share it publicly.

---

# **Block 1: Download & Install Everything**

Run this cell to download the project and install all dependencies.

This will take 2-3 minutes.

In [ ]:
NOTEBOOK_VERSION = '0.1'
NOTEBOOK_UPDATE_MARKER = '/content/notebook_update_available.txt'


import os
import sys
import subprocess
import urllib.request

BASE_DIR = '/content/videoglancer-top'
ZIP_URL = 'https://raw.githubusercontent.com/8he8/Videoglancer.top/main/videoglancer-top.zip'
ZIP_PATH = '/content/videoglancer-top.zip'
HAS_ERRORS = False
DOWNLOAD_FAILED = False

def run(cmd, desc, shell=False, timeout=120):
    print(f"  → {desc}...")
    import threading as _t
    try:
        p = subprocess.Popen(cmd, shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def _reader():
            for _l in p.stdout:
                print(f"     {_l.rstrip()}")
        _t.Thread(target=_reader, daemon=True).start()
        p.wait(timeout=timeout)
        if p.returncode == 0:
            print(f"  ✅ {desc}")
            return True
        print(f"  ⚠️  {desc} — failed (exit code {p.returncode})")
        return False
    except subprocess.TimeoutExpired:
        p.kill()
        print(f"  ⚠️  {desc} — timed out after {timeout}s")
        return False
    except FileNotFoundError:
        print(f"  ⚠️  {desc} — command not found on this system")
        return False
    except Exception as e:
        print(f"  ⚠️  {desc} — unexpected error: {e}")
        return False

# ── use uploaded zip if present ──
if os.path.exists(ZIP_PATH):
    print("📁 Found uploaded zip — using local copy")
else:
    # ── Download ──
    print("📥 Downloading videoglancer-top.ipynb...")
    try:
        req = urllib.request.Request(
            ZIP_URL,
            headers={
                'User-Agent': (
                    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                    '(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
                ),
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            },
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            with open(ZIP_PATH, 'wb') as f:
                f.write(resp.read())
        print('  ✅ Downloaded successfully')
    except Exception as e:
        print()
        print('=' * 70)
        print(f'❌  Failed to download from {ZIP_URL}')
        print('=' * 70)
        print(f'Error: {str(e)[:200]}')
        print()
        print('If the issue persists, the file may have moved.')
        print('=' * 70)
        DOWNLOAD_FAILED = True

if not DOWNLOAD_FAILED:
    # ── Extract ──
    print()
    print('📦 Extracting project...')
    if run(['unzip', '-q', '-o', ZIP_PATH, '-d', '/content/'], 'Extracting project'):
        os.chdir(BASE_DIR)
        if BASE_DIR not in sys.path:
            sys.path.insert(0, BASE_DIR)
        print('  ✅ Project extracted to /content/videoglancer-top/')
    else:
        print('  ❌ The downloaded zip appears corrupted. Try again.')
        DOWNLOAD_FAILED = True

if not DOWNLOAD_FAILED:
    # Check if this notebook is outdated by comparing against the extracted project version
    try:
        with open(os.path.join(BASE_DIR, 'static', 'notebook-version.txt'), 'r') as f:
            latest_notebook = f.read().strip()
        if latest_notebook and latest_notebook != NOTEBOOK_VERSION:
            with open(NOTEBOOK_UPDATE_MARKER, 'w') as f:
                f.write(latest_notebook)
        else:
            try: os.remove(NOTEBOOK_UPDATE_MARKER)
            except: pass
    except Exception:
        try: os.remove(NOTEBOOK_UPDATE_MARKER)
        except: pass

    # ── Python packages ──
    print()
    print('📦 Installing Python dependencies (this takes 1–2 min)...')
    if not run(['pip', 'install', '-q', '-U', '-r', 'requirements.txt'], 'Core Python packages'):
        HAS_ERRORS = True
    if not run(['pip', 'install', '-q', '-U', 'yt-dlp', 'yt-dlp-ejs'], 'yt-dlp + yt-dlp-ejs'):
        HAS_ERRORS = True

    # ── FFmpeg ──
    print()
    print('🎬 Installing FFmpeg (video processing)...')
    if not run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], 'FFmpeg'):
        HAS_ERRORS = True

    # ── Deno ──
    print()
    print('🔧 Installing Deno (YouTube challenge solver)...')
    deno_ok = run('curl -fsSL https://deno.land/x/install/install.sh | sh', 'Deno', shell=True, timeout=120)
    if deno_ok:
        deno_path = os.path.expanduser('~/.deno/bin')
        if deno_path not in os.environ['PATH']:
            os.environ['PATH'] = deno_path + os.pathsep + os.environ['PATH']
    else:
        HAS_ERRORS = True

    # ── Cloudflare Tunnel ──
    print()
    print('🌐 Installing Cloudflare Tunnel (public URL)...')
    if run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], 'Downloading cloudflared'):
        if not run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], 'Installing cloudflared'):
            HAS_ERRORS = True
    else:
        HAS_ERRORS = True

    # ── Verification ──
    print()
    print('🔍 Verifying installations...')
    checks = {
        'yt-dlp': (['yt-dlp', '--version'], 'System binary'),
        'Flask': (['python', '-c', 'import flask; print(flask.__version__)'], 'Python package'),
        'FFmpeg': (['ffmpeg', '-version'], 'System binary'),
        'Deno': (['deno', '--version'], 'System binary'),
        'Cloudflared': (['cloudflared', '--version'], 'System binary'),
    }
    missing = []
    for name, (cmd, kind) in checks.items():
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        if r.returncode == 0:
            first = r.stdout.strip().split('\n')[0][:80]
            print(f'  ✅ {name}: {first}')
        else:
            print(f'  ⚠️  {name}: not found ({kind})')
            missing.append(name)

    if missing:
        HAS_ERRORS = True
        print()
        print('⚠️  Some tools failed to install:')
        for m in missing:
            print(f'     - {m}')
        print('   Related features may not work. You can still proceed to Block 2.')

print()
if DOWNLOAD_FAILED:
    print('⚠️  Block 1 failed — could not download project. Check the URL or upload the zip manually.')
elif HAS_ERRORS:
    print('⚠️  Block 1 completed with warnings — proceed to Block 2.')
else:
    print('✅ Block 1 complete! All tools installed successfully. Move to Block 2.')




# **Block 2: Mount Google Drive**

Run this cell to mount Google Drive for persistent file storage.

This is optional but recommended to save your files permanently.

In [ ]:
from google.colab import drive
import os
import time

DRIVE_MOUNT_PATH = '/content/drive'
LOCAL_OUTPUT_DIR = '/content/videoglancer-top/output/'

print('☁️  Mounting Google Drive...')
print()

if os.path.isfile(os.path.join(DRIVE_MOUNT_PATH, 'MyDrive', '.shortcut-targets-by-id')):
    print('  ✅ Google Drive is already mounted.')
else:
    print('  A browser tab will open asking for permission.')
    print('  Click "Connect to Google Drive" and then click "Continue" to sign in with Google.')
    print('  If nothing opens, look for a popup blocker notification.\n')

    try:
        drive.mount(DRIVE_MOUNT_PATH)
        time.sleep(1)

        if os.path.isdir(os.path.join(DRIVE_MOUNT_PATH, 'MyDrive')):
            print()
            print('  ✅ Google Drive mounted successfully!')
        else:
            print()
            print('  ⚠️  Mount completed but MyDrive is not accessible.')
            print(f'  Files will be saved locally in {LOCAL_OUTPUT_DIR}')

    except Exception as e:
        err_msg = str(e).lower()
        print()

        if any(word in err_msg for word in ['cancel', 'denied', 'rejected', 'access_blocked', 'user cancelled']):
            print('  ⚠️  Google Drive mount cancelled.')
        elif any(word in err_msg for word in ['auth', 'credential', 'token', 'permission', 'forbidden']):
            print('  ⚠️  Google Drive authentication failed.')
        elif any(word in err_msg for word in ['timeout', 'connection', 'network', 'eof', 'reset']):
            print('  ⚠️  Google Drive mount timed out.')
        else:
            print('  ⚠️  Could not mount Google Drive.')
            if str(e).strip():
                print(f'  Details: {str(e)[:200]}')

        print('  No problem — files will be saved locally instead.')

print()
print('  📍 Local output: ' + LOCAL_OUTPUT_DIR)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
print('  ✅ Now proceed to Block 3.')

# **Block 3: Start the Dev Server**

Run this cell to start the dev server and get a random temporary Cloudflare Tunnel URL.

**⚠️ IMPORTANT:**
- Keep this cell running — do not stop it
- The tunnel URL will appear below in 10–30 seconds
- Click the URL to open the website

In [ ]:
import os
import subprocess
import threading
import time
import sys
import re
import socket

PROJECT_DIR = '/content/videoglancer-top'

# ── Find an available port (try 5000, 5001, ..., 5010) ──
def find_free_port(start=5000, end=5010):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return None

FLASK_PORT = find_free_port()
if FLASK_PORT is None:
    print('❌  Could not find an available port between 5000 and 5010.')
    print('   Please free up a port or restart the runtime.')
    raise SystemExit(1)

print('=' * 70)
print('🚀  STARTING DEV SERVER')
print('=' * 70)
print(f'  Using port: {FLASK_PORT}')
print()

# ── Start Flask ──
flask_started = threading.Event()

def start_flask():
    print(f'[1/2] Starting Flask server on port {FLASK_PORT}...')
    print('📋  Flask logs will appear below:\n')
    env = os.environ.copy()
    env['FLASK_DEBUG'] = '0'
    env['FLASK_USE_RELOADER'] = '0'
    env['PORT'] = str(FLASK_PORT)
    try:
        process = subprocess.Popen(
            ['python', 'app.py'],
            cwd=PROJECT_DIR, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )
        flask_started.set()
        for line in process.stdout:
            print(f'  [FLASK] {line.strip()}')
        process.wait()
    except FileNotFoundError:
        print('  ❌ Python not found. Did Step 1 complete?')
    except Exception as e:
        print(f'  ❌ Failed to start Flask: {e}')

flask_thread = threading.Thread(target=start_flask, daemon=True)
flask_thread.start()
flask_started.wait(timeout=10)

# Wait for Flask to actually bind the port
for _ in range(60):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', FLASK_PORT)) == 0:
            break
    time.sleep(0.5)

# ── Start Cloudflare Tunnel ──
print(f'\n[2/2] Starting Cloudflare Tunnel (port {FLASK_PORT})...')
print('⏳  Waiting for public URL (10–30 seconds)...\n')

try:
    tunnel_process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{FLASK_PORT}', '--loglevel', 'info'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, universal_newlines=True
    )

    public_url = None
    url_found = False
    for _ in range(200):
        try:
            line = tunnel_process.stdout.readline()
        except ValueError:
            break
        if line:
            print(f'  [TUNNEL] {line.strip()}')
            if not url_found:
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
                if match:
                    public_url = match.group(0)
                    url_found = True
                    print()
                    print()
                    print('=' * 70)
                    print('🌍  PUBLIC URL')
                    print(f'    {public_url}')
                    print('=' * 70)
                    print()
                    print('▶  Click the URL above to open the website')
                    print('▶  Keep this notebook running to maintain access')
                    print()
                    print()
                    print('🎉  WHAT TO DO NOW')
                    print()
                    print('    1. Click the public URL above')
                    print('    2. Paste a YouTube URL and click Convert')
                    print('    3. Download your PDF or PPTX file')
                    print()
                    print('⏱  Session timed out?')
                    print('     Colab sessions expire after 90 minutes')
                    print('     Just run all 3 blocks again')
                    print('=' * 70)
                    print()
                    print()
    if not public_url:
        print()
        print('=' * 70)
        print('❌  Could not detect a tunnel URL.')
        print('=' * 70)
        print(f'  The app is still running locally at http://127.0.0.1:{FLASK_PORT}')
        print('  But you cannot access it from outside Colab without a tunnel.')
        print('  In that case, run this block again.')

    tunnel_process.wait()

except FileNotFoundError:
    print()
    print('⚠️  Cloudflared not found. Run Block 1 first.')
except Exception as e:
    print()
    print(f'⚠️  Tunnel error: {str(e)[:200]}')
    print(f'  Flask is still running locally on port {FLASK_PORT}.')

# If we got here without a URL, show the fallback
if not url_found:
    print()
    print('⏳  Waiting for the public URL...')
    print('  If it does not appear within 60 seconds, restart the runtime and try again.')